# M4S1 - HPC Applications in Science, Engineering and Society

## Five small workloads, one HPC way of thinking

**Learning loop:** **predict → run → observe → discuss**

This notebook uses small, self-contained teaching kernels to represent real HPC application patterns:

| Case | Real application examples | Teaching kernel |
|---|---|---|
| Climate & weather | WRF, CESM, MPAS | 2D grid update + resolution scaling |
| CFD & engineering | OpenFOAM, SU2 | iterative 2D Poisson/pressure kernel |
| Health & genomics | BWA, GATK | chunked sequence analysis |
| Molecular & materials | GROMACS, LAMMPS | pairwise Lennard-Jones forces |
| Finance | Monte Carlo risk | vectorized scenario simulation |

> **Important:** these are simplified teaching kernels, not replacements for production applications.  
> The objective is to recognize the **computational pattern, parallelism, bottleneck, and useful metric**.

> **Environment note:** this version uses NumPy plus built-in Jupyter SVG/HTML. It does not require Matplotlib.


## 0 - Check the environment

Before doing science, check **where** the notebook is running and which software environment it is using.

On an HPC system, the software stack matters: Python version, numerical libraries, compiler/runtime, scheduler, and available modules.

In [ ]:
import os, sys, subprocess, shutil, time, math, html
from pathlib import Path
from collections import Counter

import numpy as np
from IPython.display import display, HTML, SVG

def shell(cmd):
    result = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    return (result.stdout + result.stderr).strip()

print("Hostname:", shell("hostname"))
print("Python:", sys.version.split()[0])
print("Executable:", sys.executable)
print("CPU cores visible:", os.cpu_count())
print("NumPy:", np.__version__)
print("Slurm sbatch available:", shutil.which("sbatch") is not None)

print("\nLoaded modules / module system:")
print(shell("bash -lc 'if type module >/dev/null 2>&1; then module list 2>&1; else echo \"module command not visible in this shell\"; fi'"))

def _esc(s):
    return html.escape(str(s))

def _svg_show(body, width=760, height=420):
    display(SVG(f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">{body}</svg>'))

def _value_color(v, vmin, vmax):
    t = 0.5 if vmax <= vmin else max(0.0, min(1.0, (float(v)-float(vmin))/(float(vmax)-float(vmin))))
    if t < 1/3:
        q=t*3; r,g,b=30,int(80+150*q),int(220-30*q)
    elif t < 2/3:
        q=(t-1/3)*3; r,g,b=int(30+220*q),int(230+20*q),int(190-150*q)
    else:
        q=(t-2/3)*3; r,g,b=250,int(250-180*q),int(40-20*q)
    return f'rgb({r},{g},{b})'

def heatmap_svg(a, title="", sample=55, width=720, height=420):
    a=np.asarray(a)
    sy=max(1,a.shape[0]//sample); sx=max(1,a.shape[1]//sample)
    small=a[::sy,::sx][:sample,:sample]
    h,w=small.shape; left,top=65,50; plot_w,plot_h=width-100,height-90
    cw,ch=plot_w/w,plot_h/h
    vmin,vmax=float(np.min(small)),float(np.max(small))
    parts=[f'<rect width="{width}" height="{height}" fill="white"/>',
           f'<text x="{width/2}" y="28" text-anchor="middle" font-family="sans-serif" font-size="20" font-weight="600">{_esc(title)}</text>']
    for iy in range(h):
        for ix in range(w):
            parts.append(f'<rect x="{left+ix*cw:.2f}" y="{top+iy*ch:.2f}" width="{cw+0.3:.2f}" height="{ch+0.3:.2f}" fill="{_value_color(small[iy,ix],vmin,vmax)}"/>')
    parts.append(f'<text x="{left}" y="{height-12}" font-family="sans-serif" font-size="12">min {vmin:.3g}</text>')
    parts.append(f'<text x="{width-135}" y="{height-12}" font-family="sans-serif" font-size="12">max {vmax:.3g}</text>')
    _svg_show("".join(parts),width,height)

def line_svg(xs, ys, title="", xlabel="", ylabel="", width=720, height=390):
    xs=np.asarray(xs,dtype=float); ys=np.asarray(ys,dtype=float)
    l,r,t,b=80,30,52,60; pw,ph=width-l-r,height-t-b
    xmin,xmax=float(xs.min()),float(xs.max()); ymin,ymax=float(ys.min()),float(ys.max())
    if xmax==xmin: xmax+=1
    if ymax==ymin: ymax+=1
    X=l+(xs-xmin)/(xmax-xmin)*pw; Y=t+ph-(ys-ymin)/(ymax-ymin)*ph
    pts=" ".join(f"{x:.1f},{y:.1f}" for x,y in zip(X,Y))
    parts=[f'<rect width="{width}" height="{height}" fill="white"/>',
           f'<text x="{width/2}" y="28" text-anchor="middle" font-family="sans-serif" font-size="20" font-weight="600">{_esc(title)}</text>',
           f'<line x1="{l}" y1="{t}" x2="{l}" y2="{t+ph}" stroke="#555"/>',
           f'<line x1="{l}" y1="{t+ph}" x2="{l+pw}" y2="{t+ph}" stroke="#555"/>',
           f'<polyline fill="none" stroke="#2463a6" stroke-width="3" points="{pts}"/>']
    for x,y,xv,yv in zip(X,Y,xs,ys):
        parts += [f'<circle cx="{x:.1f}" cy="{y:.1f}" r="5" fill="#2463a6"/>',
                  f'<text x="{x:.1f}" y="{y-10:.1f}" text-anchor="middle" font-family="sans-serif" font-size="11">{yv:.3g}</text>',
                  f'<text x="{x:.1f}" y="{t+ph+20:.1f}" text-anchor="middle" font-family="sans-serif" font-size="11">{xv:g}</text>']
    parts += [f'<text x="{l+pw/2}" y="{height-12}" text-anchor="middle" font-family="sans-serif" font-size="13">{_esc(xlabel)}</text>',
              f'<text transform="translate(18,{t+ph/2}) rotate(-90)" text-anchor="middle" font-family="sans-serif" font-size="13">{_esc(ylabel)}</text>']
    _svg_show("".join(parts),width,height)

def bars_svg(labels, values, title="", width=720, height=390):
    values=np.asarray(values,dtype=float); l,r,t,b=60,25,52,70
    pw,ph=width-l-r,height-t-b; vmax=max(float(values.max()),1e-12); n=len(values)
    gap=pw/max(n,1)*0.20; bw=pw/max(n,1)-gap
    parts=[f'<rect width="{width}" height="{height}" fill="white"/>',
           f'<text x="{width/2}" y="28" text-anchor="middle" font-family="sans-serif" font-size="20" font-weight="600">{_esc(title)}</text>',
           f'<line x1="{l}" y1="{t+ph}" x2="{l+pw}" y2="{t+ph}" stroke="#555"/>']
    for i,(lab,v) in enumerate(zip(labels,values)):
        bh=float(v)/vmax*ph; x=l+i*(bw+gap)+gap/2; y=t+ph-bh
        parts += [f'<rect x="{x:.1f}" y="{y:.1f}" width="{bw:.1f}" height="{bh:.1f}" fill="#4b7f9e"/>',
                  f'<text x="{x+bw/2:.1f}" y="{t+ph+20}" text-anchor="middle" font-family="sans-serif" font-size="11">{_esc(lab)}</text>']
    _svg_show("".join(parts),width,height)

def histogram_svg(values, bins=50, title="", marker=None, marker_label="", width=760, height=390):
    counts,edges=np.histogram(values,bins=bins); l,r,t,b=65,25,52,65
    pw,ph=width-l-r,height-t-b; vmax=max(int(counts.max()),1); bw=pw/len(counts)
    parts=[f'<rect width="{width}" height="{height}" fill="white"/>',
           f'<text x="{width/2}" y="28" text-anchor="middle" font-family="sans-serif" font-size="20" font-weight="600">{_esc(title)}</text>',
           f'<line x1="{l}" y1="{t+ph}" x2="{l+pw}" y2="{t+ph}" stroke="#555"/>']
    for i,c in enumerate(counts):
        bh=c/vmax*ph
        parts.append(f'<rect x="{l+i*bw:.1f}" y="{t+ph-bh:.1f}" width="{max(1,bw-0.5):.1f}" height="{bh:.1f}" fill="#5f8f72"/>')
    if marker is not None:
        xmin,xmax=float(edges[0]),float(edges[-1]); mx=l+(float(marker)-xmin)/(xmax-xmin)*pw
        parts += [f'<line x1="{mx:.1f}" y1="{t}" x2="{mx:.1f}" y2="{t+ph}" stroke="#b74343" stroke-width="3" stroke-dasharray="6,5"/>',
                  f'<text x="{mx+6:.1f}" y="{t+16}" font-family="sans-serif" font-size="12" fill="#8f2f2f">{_esc(marker_label)}</text>']
    parts += [f'<text x="{l}" y="{t+ph+20}" font-family="sans-serif" font-size="11">{edges[0]:.2f}</text>',
              f'<text x="{l+pw-45}" y="{t+ph+20}" font-family="sans-serif" font-size="11">{edges[-1]:.2f}</text>']
    _svg_show("".join(parts),width,height)

print("\nVisualization helpers ready. Matplotlib is not required.")


## Real HPC software stack: application vs library vs runtime

The small examples in this notebook are teaching kernels. Real HPC work normally combines several software layers:

| Layer | Examples | What it does |
|---|---|---|
| **Domain application** | WRF, OpenFOAM, BWA-MEM2, GROMACS, LAMMPS, Quantum ESPRESSO | solves the scientific or engineering problem |
| **Numerical / data library** | BLAS, LAPACK, FFTW, PETSc, HDF5, NetCDF | provides optimized building blocks |
| **Parallel runtime** | MPI, OpenMP, CUDA, NCCL | executes work in parallel |
| **Scheduler / workflow** | Slurm, job arrays, containers, pipelines | places and manages work on the cluster |
| **Python ecosystem** | NumPy, SciPy, xarray, PyTorch, Dask | high-level interface to optimized compiled code |

**Key idea:** HPC applications are rarely one program doing everything. They are a stack.

```text
scientific problem
      ↓
domain application
      ↓
numerical + data libraries
      ↓
MPI / OpenMP / GPU runtime
      ↓
Slurm
      ↓
CPU / GPU nodes + storage
```


## Live cluster software reconnaissance

Let us ask the cluster what real scientific applications are available.

The cell below does not assume that a package is installed. It searches the module system and reports what it can find.

This is useful because HPC software is normally provided through environment modules rather than installed separately by every user.


In [ ]:
import re, shlex

def module_avail(term, limit=8):
    cmd = f"bash -lc 'module -t avail {shlex.quote(term)} 2>&1 | head -n {limit}'"
    p = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    text = (p.stdout + p.stderr).strip()
    text = re.sub(r"\x1b\[[0-9;]*m", "", text)
    return text or "(no match shown)"

software_queries = {
    "Weather / climate": ["wrf", "netcdf"],
    "CFD / engineering": ["openfoam", "petsc", "paraview"],
    "Genomics": ["bwa", "samtools"],
    "Molecular / materials": ["gromacs", "lammps", "quantumespresso"],
    "GPU / AI stack": ["nvhpc", "cuda"],
    "Finance / numerical kernels": ["OpenBLAS"],
}

for domain, terms in software_queries.items():
    print("\n" + "="*70)
    print(domain)
    print("="*70)
    for term in terms:
        print(f"\nmodule avail {term}")
        print(module_avail(term))


### Helper for the real module-backed examples

The cluster exposes scientific applications and libraries through environment modules.

This helper searches for the first matching module, loads its exact module name, and then runs the requested command. This avoids assuming that a generic command such as `module load lammps` will work.

In [ ]:
import re, shlex, subprocess, os, time
from pathlib import Path

def find_module(term):
    p = subprocess.run(
        ["bash", "-lc", f"module -t avail {shlex.quote(term)} 2>&1"],
        text=True, capture_output=True
    )
    text = re.sub(r"\x1b\[[0-9;]*m", "", p.stdout + p.stderr)
    candidates = []
    needle = term.lower()
    for raw in text.splitlines():
        line = raw.strip()
        if not line or line.startswith("/") or line.startswith("-"):
            continue
        if "/" in line and needle in line.lower():
            candidates.append(line.split()[0])
    return candidates[0] if candidates else None

def run_with_module(term, command, timeout=120):
    mod = find_module(term)
    if not mod:
        return f"NO_MODULE_FOUND:{term}"
    script = f"""set -e
module purge >/dev/null 2>&1 || true
module load {shlex.quote(mod)}
echo "MODULE_LOADED={mod}"
{command}
"""
    p = subprocess.run(["bash","-lc",script], text=True, capture_output=True, timeout=timeout)
    return (p.stdout + p.stderr).strip()

def run_with_modules(terms, command, timeout=120):
    mods = []
    for term in terms:
        mod = find_module(term)
        if not mod:
            return f"NO_MODULE_FOUND:{term}"
        mods.append(mod)
    loads = "\n".join(f"module load {shlex.quote(m)}" for m in mods)
    script = f"""set -e
module purge >/dev/null 2>&1 || true
{loads}
echo "MODULES_LOADED={' | '.join(mods)}"
{command}
"""
    p = subprocess.run(["bash","-lc",script], text=True, capture_output=True, timeout=timeout)
    return (p.stdout + p.stderr).strip()

print("Module helper ready.")

## A real HPC library is already being used

The teaching kernels use **NumPy**. NumPy is not just "Python loops": many operations call optimized compiled numerical code underneath.

Typical HPC libraries beneath scientific Python include:

- **BLAS / LAPACK** for dense linear algebra
- optimized vector kernels
- FFT implementations
- threaded math libraries

Run the next cell and inspect the build configuration. The exact backend depends on the cluster environment.


In [ ]:
import io, contextlib

buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    np.show_config()

cfg = buf.getvalue()
print(cfg[:5000])

# Small example: dense matrix multiplication uses compiled numerical kernels.
rng = np.random.default_rng(42)
A = rng.random((1200, 1200))
B = rng.random((1200, 1200))

t0 = time.perf_counter()
C = A @ B
elapsed = time.perf_counter() - t0

print(f"\nMatrix multiplication: 1200 x 1200")
print(f"Runtime: {elapsed:.3f} s")
print(f"Checksum: {C.sum():.3e}")
print("\nThis is the same principle used by much larger simulation and AI applications:")
print("high-level code -> optimized numerical library -> CPU/GPU hardware")


## The five-question case canvas

For every case, ask:

1. **DOMAIN / PROBLEM** - What are we trying to achieve?
2. **COMPUTATION & DATA** - What must be calculated or processed?
3. **PARALLEL PATTERN** - What can happen simultaneously?
4. **BOTTLENECK** - Compute? Memory? Communication? I/O?
5. **METRIC** - What does “fast enough” mean here?

We will use the same canvas five times.

# Case 1 - Climate & weather

### Predict before running
If we make a weather grid **twice as fine in each horizontal direction**, do you expect the computational work to:
- roughly stay the same,
- roughly double,
- or grow much faster?

A production weather model has many variables, vertical levels, physics packages, communication, and I/O.  
Here we use a simple 2D diffusion-style grid update to expose the basic **grid × timestep** pattern.

In [ ]:
def climate_initial(n=256, seed=0):
    y, x = np.mgrid[0:n, 0:n]
    rng = np.random.default_rng(seed)
    cx, cy = n * 0.45, n * 0.55
    sigma = n * 0.08
    field = np.exp(-((x-cx)**2 + (y-cy)**2) / (2*sigma**2))
    field += 0.05 * rng.random((n, n))
    return field

def climate_stepper(field, steps=120, alpha=0.18):
    u = field.copy()
    for _ in range(steps):
        old = u.copy()
        u[1:-1, 1:-1] = old[1:-1, 1:-1] + alpha * (
            old[2:, 1:-1] + old[:-2, 1:-1] +
            old[1:-1, 2:] + old[1:-1, :-2] -
            4 * old[1:-1, 1:-1]
        )
    return u

n = 320
initial = climate_initial(n)
t0 = time.perf_counter()
final = climate_stepper(initial, steps=160)
elapsed = time.perf_counter() - t0

heatmap_svg(initial, "Climate teaching field - initial")
heatmap_svg(final, f"Climate teaching field - after 160 updates ({elapsed:.3f} s)")
print(f"Grid: {n} x {n} = {n*n:,} cells")
print(f"Elapsed: {elapsed:.3f} s")


In [ ]:
sizes = [128, 256, 512, 768]
times = []

for n in sizes:
    u = climate_initial(n)
    t0 = time.perf_counter()
    _ = climate_stepper(u, steps=80)
    times.append(time.perf_counter() - t0)

line_svg(sizes, times,
         "Climate teaching kernel - cost grows with resolution",
         "Grid width N (N x N cells)", "Runtime (s)")


### Observe / discuss

**Case canvas**
- **Problem:** forecast weather / simulate climate at high resolution
- **Computation:** grid cells × variables × timesteps; often many ensemble members
- **Parallel pattern:** domain decomposition + ensemble parallelism
- **Likely bottlenecks:** communication, memory bandwidth, I/O
- **Useful metric:** **time-to-forecast / ensemble throughput**

**Discuss:** Why can a scientifically correct forecast still be operationally useless?

**Real-world mapping:** WRF, CESM and MPAS are vastly more sophisticated, but the same pressure remains: **more resolution → more computation and data → more need for parallel HPC**.

# Case 2 - CFD & engineering

### Predict before running
A CFD mesh is split across several MPI ranks.

**Question:** Which cells normally require communication with another rank?

The kernel below is a simplified iterative pressure/Poisson solve - a common numerical pattern inside CFD workflows.

In [ ]:
def cfd_poisson(n=220, iterations=300):
    p = np.zeros((n, n), dtype=float)
    source = np.zeros_like(p)
    source[n//4, n//4] = 100.0
    source[3*n//4, 3*n//4] = -100.0
    dx = 1.0 / (n - 1)

    for _ in range(iterations):
        old = p.copy()
        p[1:-1, 1:-1] = (
            old[1:-1, 2:] + old[1:-1, :-2] +
            old[2:, 1:-1] + old[:-2, 1:-1] -
            source[1:-1, 1:-1] * dx * dx
        ) / 4.0
    return p

n = 240
t0 = time.perf_counter()
pressure = cfd_poisson(n, 320)
elapsed = time.perf_counter() - t0

heatmap_svg(pressure, f"Simplified CFD pressure field ({n} x {n}, {elapsed:.3f} s)")
print("Conceptual decomposition:")
print("[ Rank 0 ][ Rank 1 ]")
print("[ Rank 2 ][ Rank 3 ]")
print("Halo exchange is needed near shared boundaries.")


In [ ]:
sizes = [100, 180, 260, 360]
times = []

for n in sizes:
    t0 = time.perf_counter()
    _ = cfd_poisson(n, 180)
    times.append(time.perf_counter() - t0)

line_svg(sizes, times,
         "CFD teaching kernel - finer mesh costs more",
         "Mesh width N", "Runtime (s)")


### Observe / discuss

**Case canvas**
- **Problem:** airflow around a car, wing, turbine, building, etc.
- **Computation:** mesh + repeated numerical updates
- **Parallel pattern:** domain decomposition; MPI between subdomains; OpenMP/threads inside a node
- **Likely bottlenecks:** halo communication, memory bandwidth, load imbalance
- **Useful metric:** **time-to-solution / design iterations per day**

**Discuss:** Why can adding MPI ranks eventually stop helping?

**Real-world mapping:** OpenFOAM, SU2 and commercial CFD solvers use much richer discretizations and physics, but domain decomposition and communication remain central HPC ideas.

# Case 3 - Health & genomics

### Predict before running
A sequencing study contains **many independent reads or samples**.

Which pattern is more natural first?
- spatial domain decomposition,
- or **data/task parallelism**?

We will generate synthetic reads, split them into chunks, and reduce partial results.

In [ ]:
def generate_reads(n_reads=18_000, read_len=100, seed=4):
    rng = np.random.default_rng(seed)
    bases = np.array(list("ACGT"))
    arr = rng.choice(bases, size=(n_reads, read_len),
                     p=[0.30, 0.20, 0.20, 0.30])
    return ["".join(row) for row in arr]

def analyse_chunk(reads):
    base_counts = Counter()
    kmer_counts = Counter()
    for read in reads:
        base_counts.update(read)
        kmer_counts.update(read[i:i+3] for i in range(len(read)-2))
    return base_counts, kmer_counts

reads = generate_reads()
chunks = np.array_split(np.array(reads, dtype=object), 4)

t0 = time.perf_counter()
partials = [analyse_chunk(list(chunk)) for chunk in chunks]
elapsed = time.perf_counter() - t0

base_total = sum((p[0] for p in partials), Counter())
kmer_total = sum((p[1] for p in partials), Counter())

bars_svg(["A","C","G","T"], [base_total[b] for b in "ACGT"],
         "Synthetic sequencing - base composition")
top = kmer_total.most_common(10)
bars_svg([k for k,v in top], [v for k,v in top],
         "Synthetic sequencing - top 3-mers")

print(f"Reads analysed: {len(reads):,}")
print(f"Elapsed: {elapsed:.3f} s")
print("Pattern: READ CHUNKS -> independent analysis -> MERGE / REDUCE")


In [ ]:
single_bases, single_kmers = analyse_chunk(reads)

print("Base counts identical:", single_bases == base_total)
print("3-mer counts identical:", single_kmers == kmer_total)
print("\nParallel pattern:")
print("READ CHUNKS -> independent analysis -> REDUCE / MERGE")

### Observe / discuss

**Case canvas**
- **Problem:** identify biological information from sequencing data
- **Computation:** read processing → alignment/analysis → variant calling
- **Parallel pattern:** data parallelism + pipeline stages
- **Likely bottlenecks:** I/O, memory, pipeline imbalance
- **Useful metric:** **samples/day / turnaround time**

**Discuss:** Why does equal file size not always imply equal runtime?

**Real-world mapping:** tools such as BWA and GATK perform much more sophisticated algorithms, but large genomics workflows still expose **chunking, pipelines, data movement, and throughput** as HPC concerns.

# Case 4 - Molecular simulation & materials

### Predict before running
If every particle interacts with every other particle, what happens to the amount of work when the number of particles doubles?

We will compute simplified **Lennard-Jones pair forces**, one of the classic ideas used to teach molecular simulation.

In [ ]:
def make_particles(n=180, density=0.50, seed=2):
    box = math.sqrt(n / density)
    side = math.ceil(math.sqrt(n))
    spacing = box / side
    xs = (np.arange(side) + 0.5) * spacing
    points = np.array(np.meshgrid(xs, xs)).reshape(2, -1).T[:n]
    rng = np.random.default_rng(seed)
    points += rng.normal(scale=0.03 * spacing, size=points.shape)
    return points, box

def lennard_jones_forces(pos, box, cutoff=3.0):
    delta = pos[:, None, :] - pos[None, :, :]
    delta -= box * np.round(delta / box)
    r2 = np.sum(delta**2, axis=-1)
    np.fill_diagonal(r2, np.inf)

    mask = r2 < cutoff**2
    inv_r2 = np.zeros_like(r2)
    inv_r2[mask] = 1.0 / r2[mask]
    inv_r6 = inv_r2**3
    inv_r12 = inv_r6**2
    scalar = np.zeros_like(r2)
    scalar[mask] = 24.0 * (2.0*inv_r12[mask] - inv_r6[mask]) * inv_r2[mask]
    return np.sum(scalar[..., None] * delta, axis=1)

pos, box = make_particles()
t0 = time.perf_counter()
forces = lennard_jones_forces(pos, box)
elapsed = time.perf_counter() - t0

W=650; H=650; pad=35
parts=[f'<rect width="{W}" height="{H}" fill="white"/>',
       f'<text x="{W/2}" y="25" text-anchor="middle" font-family="sans-serif" font-size="20" font-weight="600">Simplified molecular force field</text>']
mags=np.linalg.norm(forces,axis=1)
scale=max(float(np.percentile(mags,90)),1e-12)
for (x,y),(fx,fy) in zip(pos,forces):
    sx=pad+x/box*(W-2*pad); sy=H-pad-y/box*(H-2*pad)
    ex=sx+(fx/scale)*13; ey=sy-(fy/scale)*13
    parts.append(f'<line x1="{sx:.1f}" y1="{sy:.1f}" x2="{ex:.1f}" y2="{ey:.1f}" stroke="#789" stroke-width="1"/>')
    parts.append(f'<circle cx="{sx:.1f}" cy="{sy:.1f}" r="3.2" fill="#315d86"/>')
_svg_show("".join(parts),W,H)
print(f"Particles: {len(pos)}")
print(f"Force calculation runtime: {elapsed:.4f} s")


In [ ]:
sizes = [100, 200, 400, 700]
times = []

for n in sizes:
    pos, box = make_particles(n)
    t0 = time.perf_counter()
    _ = lennard_jones_forces(pos, box)
    times.append(time.perf_counter() - t0)

line_svg(sizes, times,
         "Naive pair-force kernel - increasing work",
         "Particles", "Runtime (s)")


### Observe / discuss

**Case canvas**
- **Problem:** predict how atoms/molecules behave
- **Computation:** force evaluation → update positions → repeat
- **Parallel pattern:** spatial decomposition + accelerators
- **Likely bottlenecks:** force computation, communication, memory
- **Useful metric:** **simulated time/day (e.g. ns/day) / time-to-solution**

**Discuss:** Why are GPUs attractive for force calculations?

**Real-world mapping:** GROMACS and LAMMPS add advanced neighbor lists, decomposition, vectorization, GPUs, communication and sophisticated physical models. The small kernel above is only the computational idea.

# Case 5 - Finance: Monte Carlo at scale

### Predict before running
Suppose one risk scenario takes 1 millisecond.

What changes when you need **10 million independent scenarios before a decision deadline**?

This is an example of **embarrassingly parallel work followed by a reduction/statistical summary**.

In [ ]:
def finance_monte_carlo(n=400_000, seed=7):
    rng = np.random.default_rng(seed)
    S0, mu, sigma, T = 100.0, 0.05, 0.22, 1.0
    z = rng.standard_normal(n)
    terminal = S0 * np.exp((mu - 0.5*sigma**2)*T + sigma*np.sqrt(T)*z)
    pnl = terminal - S0
    return pnl, float(np.quantile(pnl, 0.05))

t0 = time.perf_counter()
pnl, var95 = finance_monte_carlo()
elapsed = time.perf_counter() - t0

histogram_svg(pnl, bins=60, title="Monte Carlo risk distribution",
              marker=var95, marker_label=f"5th percentile = {var95:.2f}")
print(f"Scenarios: {len(pnl):,}")
print(f"Runtime: {elapsed:.3f} s")
print(f"Throughput: {len(pnl)/elapsed:,.0f} scenarios/s")


In [ ]:
N = 60_000
rng = np.random.default_rng(9)
z = rng.standard_normal(N)
S0, mu, sigma, T = 100.0, 0.05, 0.22, 1.0

t0 = time.perf_counter()
vec = S0 * np.exp((mu - 0.5*sigma**2)*T + sigma*np.sqrt(T)*z)
t_vec = time.perf_counter() - t0

t0 = time.perf_counter()
loop = np.empty(N)
for i, zi in enumerate(z):
    loop[i] = S0 * math.exp((mu - 0.5*sigma**2)*T + sigma*math.sqrt(T)*zi)
t_loop = time.perf_counter() - t0

print(f"Vectorized NumPy: {t_vec:.4f} s")
print(f"Python loop:      {t_loop:.4f} s")
print(f"Observed ratio:   {t_loop/max(t_vec,1e-12):.1f}x")
print("Results agree:", np.allclose(vec, loop))


### Observe / discuss

**Case canvas**
- **Problem:** estimate portfolio/risk outcomes before a decision deadline
- **Computation:** many independent scenarios
- **Parallel pattern:** embarrassingly parallel → aggregate/reduce
- **Likely bottlenecks:** throughput, data movement, aggregation
- **Useful metric:** **scenarios/second / time-to-decision**

**Discuss:** How is this fundamentally different from CFD?

- CFD subdomains often exchange boundary information repeatedly.
- Monte Carlo scenarios can often run independently until final aggregation.

# More real HPC applications to recognise

The same HPC ideas appear in many other domains:

| Domain | Real applications / frameworks | Typical HPC pattern |
|---|---|---|
| **Weather & climate** | WRF, CESM, MPAS, ICON | grids, MPI domain decomposition, ensembles, NetCDF |
| **CFD & engineering** | OpenFOAM, SU2, Nek5000 / NekRS | meshes, sparse solvers, halo exchange, GPUs |
| **Genomics** | BWA-MEM2, samtools, GATK, Nextflow | data parallelism, pipelines, large I/O |
| **Molecular dynamics** | GROMACS, LAMMPS, NAMD | particle decomposition, MPI, SIMD, GPUs |
| **Materials / quantum chemistry** | Quantum ESPRESSO, CP2K, GPAW | FFTs, dense/sparse linear algebra, MPI |
| **AI training** | PyTorch, TensorFlow, NCCL, Horovod / DeepSpeed | data/model parallelism, GPU collectives |
| **Astrophysics** | GADGET, FLASH, AMReX-based codes | particles + adaptive meshes + MPI |
| **Earthquakes / seismic** | SPECFEM | large meshes, wave propagation, MPI |
| **Finance / risk** | Monte Carlo engines, NumPy/Numba, distributed task frameworks | independent scenarios + reduction |
| **Visualization** | ParaView, VisIt | parallel post-processing and large-data rendering |

### Connect the layers

For any application, ask:

**What is the domain app? Which numerical libraries does it call? How is it parallelized? How is it launched on the cluster? What metric tells us that the run is useful?**


# Compare the five cases

| Case | Main pattern | Likely bottleneck | Useful metric |
|---|---|---|---|
| Climate | domain + ensemble | communication / memory / I/O | time-to-forecast |
| CFD | domain decomposition | halos / memory / balance | time-to-solution |
| Genomics | data + pipeline | I/O / memory / imbalance | samples/day |
| Molecular | spatial + accelerator | compute / communication | simulated ns/day |
| Finance | independent tasks + reduction | throughput / aggregation | scenarios/s |

> **Peak FLOPS is a system metric. Success is an application metric.**

# Optional cluster extension - submit the five cases through Slurm

The interactive examples above are deliberately small.  
This section shows how the **same notebook can create a real batch job array**.

Each array task represents a different application case.

> Run this only if `sbatch` is available from the Jupyter environment.  
> If not, copy the generated files to your normal SSH/Slurm terminal.

In [ ]:
runner = '\nimport sys, time, math\nimport numpy as np\n\ncase = sys.argv[1]\nt0 = time.perf_counter()\n\nif case == "climate":\n    n = 768\n    u = np.random.default_rng(1).random((n,n))\n    alpha = 0.18\n    for _ in range(150):\n        old = u.copy()\n        u[1:-1,1:-1] = old[1:-1,1:-1] + alpha * (\n            old[2:,1:-1] + old[:-2,1:-1] +\n            old[1:-1,2:] + old[1:-1,:-2] -\n            4*old[1:-1,1:-1]\n        )\n    metric = f"cells={n*n:,}"\n\nelif case == "cfd":\n    n = 500\n    p = np.zeros((n,n))\n    b = np.zeros_like(p)\n    b[n//4,n//4] = 100.0\n    b[3*n//4,3*n//4] = -100.0\n    dx = 1/(n-1)\n    for _ in range(300):\n        old = p.copy()\n        p[1:-1,1:-1] = (\n            old[1:-1,2:] + old[1:-1,:-2] +\n            old[2:,1:-1] + old[:-2,1:-1] -\n            b[1:-1,1:-1]*dx*dx\n        ) / 4\n    metric = f"mesh={n}x{n}"\n\nelif case == "genomics":\n    rng = np.random.default_rng(3)\n    bases = np.array(list("ACGT"))\n    arr = rng.choice(bases, size=(120_000,120))\n    counts = {b:int(np.count_nonzero(arr == b)) for b in bases}\n    metric = f"reads={len(arr):,}; A={counts[\'A\']:,}"\n\nelif case == "molecular":\n    n = 1200\n    box = math.sqrt(n/0.5)\n    pos = np.random.default_rng(4).random((n,2))*box\n    delta = pos[:,None,:] - pos[None,:,:]\n    delta -= box*np.round(delta/box)\n    r2 = np.sum(delta**2,axis=-1)\n    np.fill_diagonal(r2,np.inf)\n    mask = r2 < 3.0**2\n    metric = f"particles={n:,}; neighbor_pairs={int(mask.sum()):,}"\n\nelif case == "finance":\n    rng = np.random.default_rng(5)\n    n = 4_000_000\n    z = rng.standard_normal(n)\n    terminal = 100*np.exp((0.05-0.5*0.22**2) + 0.22*z)\n    var5 = np.quantile(terminal-100,0.05)\n    metric = f"scenarios={n:,}; p5={var5:.3f}"\n\nelse:\n    raise SystemExit("Unknown case")\n\nelapsed = time.perf_counter()-t0\nprint(f"CASE={case} | {metric} | runtime={elapsed:.4f}s")\n'
Path("s16_case_runner.py").write_text(runner)
print("Wrote s16_case_runner.py")

In [ ]:
python_exe = sys.executable
workdir = str(Path.cwd())

sbatch_text = f'''#!/bin/bash -l
#SBATCH --job-name=s16_apps
#SBATCH --array=0-4
#SBATCH --cpus-per-task=1
#SBATCH --mem=1G
#SBATCH --time=00:05:00
#SBATCH --output=s16_case_%A_%a.out

CASES=(climate cfd genomics molecular finance)
CASE=${{CASES[$SLURM_ARRAY_TASK_ID]}}

echo "Host: $(hostname)"
echo "Case: $CASE"

if type module >/dev/null 2>&1; then
    module list 2>&1 || true
fi

cd "{workdir}"
"{python_exe}" s16_case_runner.py "$CASE"
'''

Path("s16_cases.sbatch").write_text(sbatch_text)
print(Path("s16_cases.sbatch").read_text())

In [ ]:
SUBMIT = False

if SUBMIT:
    if shutil.which("sbatch"):
        print(shell("sbatch s16_cases.sbatch"))
    else:
        print("sbatch is not available here. Submit s16_cases.sbatch from the normal cluster terminal.")
else:
    print("SUBMIT=False: no jobs submitted.")
    print("When ready, set SUBMIT=True and rerun this cell.")

In [ ]:
outputs = sorted(Path(".").glob("s16_case_*.out"))

if not outputs:
    print("No Slurm outputs found yet.")
else:
    for f in outputs:
        print(f"\n--- {f.name} ---")
        print(f.read_text().strip())

# Final discussion / exit question

Choose **one new application not used today**:

- energy-grid simulation,
- digital twin,
- earthquake modelling,
- astrophysics,
- AI training,
- another example of your choice.

Apply the five-question canvas:

**DOMAIN → COMPUTATION & DATA → PARALLEL PATTERN → BOTTLENECK → METRIC**

### Final takeaway

**HPC is not simply “a faster computer.”**  
It is a way to combine algorithms, parallelism, architecture, software and infrastructure so that a useful scientific or engineering result arrives at the required scale and time.